# 🏠 Forecasting Household Energy Consumption
## Notebook 1 — Exploratory Data Analysis (EDA)

**Capstone Project | UMBC Data Science Master's Program**
Instructor: Dr. Chaojie (Jay) Wang | Author: Kushal Erramilli

---

### What This Notebook Does
This notebook answers three core questions before any model is built:

1. **What does the data look like?** — shape, types, missing values, distribution
2. **Are there patterns we can exploit?** — hourly, daily, weekly, seasonal cycles
3. **Which variables matter most?** — correlations, sub-meter contributions, outliers


**Dataset:** UCI Individual Household Electric Power Consumption
**Path:** `../data/household_power_consumption.csv`


## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("✅ All libraries loaded successfully.")


## 1. Load Dataset

**Why:** We need to understand raw structure before cleaning anything.
The separator is `;` (not `,`) and missing entries are encoded as `?`.


In [ ]:
DATA_PATH = "../data/household_power_consumption.csv"

df_raw = pd.read_csv(
    DATA_PATH,
    na_values=["?"],
    low_memory=False
)

print(f"Shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()


## 2. Dataset Overview

**Why:** `.info()` reveals data types immediately — all power columns land as `object`
because of the `?` placeholders. We must coerce them to `float64`.


In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T


## 3. Data Cleaning

### 3.1 Combine Date + Time → DatetimeIndex

**Why:** A proper `DatetimeIndex` lets us use Pandas time-aware operations
(`.resample()`, `.shift()`, `.rolling()`) which are essential for feature
engineering in Notebook 2.


In [ ]:
df = df_raw.copy()
df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    dayfirst=True
)

df = df.sort_values("Datetime").set_index("Datetime")
df.drop(columns=["Date", "Time"], inplace=True)

numeric_cols = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"✅ Datetime index set | Numeric columns converted")
print(f"Date range : {df.index.min()}  →  {df.index.max()}")
print(f"Shape      : {df.shape}")
df.head()


### 3.2 Missing Value Analysis

**Why:** Knowing *how much* data is missing and *whether it's random or
block-structured* determines the right imputation strategy.

**What we find:** Exactly **25,979 rows** (≈ 1.25 %) are missing across all
7 sensor columns simultaneously — meaning entire one-minute windows are
missing, not individual sensors. This is typical of sensor dropouts or
communication failures rather than random corruption.


In [ ]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(3)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %"    : missing_pct
}).sort_values("Missing %", ascending=False)

print(missing_df)
print(f"\nTotal missing cells : {missing.sum():,} / {df.size:,}  "
      f"({missing.sum()/df.size*100:.2f}%)")


In [ ]:
fig = px.bar(
    missing_df[missing_df["Missing Count"] > 0].reset_index(),
    x="index", y="Missing %",
    text="Missing %",
    title="Missing Values per Column  (all columns equally affected → block dropout)",
    labels={"index": "Column"},
    color="Missing %",
    color_continuous_scale="Reds",
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(title_x=0.5, height=450, coloraxis_showscale=False)
fig.show()


### 3.3 Handle Missing Values

**Strategy:** Forward-fill (`ffill`) propagates the last valid reading into
each gap — the correct choice for sensor time series because power consumption
doesn't jump discontinuously. A back-fill guard handles any leading NaNs.

**Decision for modelling:** Because gaps are contiguous blocks (not random
scatter), interpolation would hallucinate energy data. Forward-fill is
conservative and safe.


In [ ]:
df_clean = df.copy()
df_clean.ffill(inplace=True)
df_clean.bfill(inplace=True)

print(f"✅ Missing values remaining : {df_clean.isnull().sum().sum()}")
print(f"   Clean dataset shape      : {df_clean.shape}")


## 4. Feature Engineering — Calendar Attributes

**Why:** Time-of-day, day-of-week, and season are some of the strongest
predictors of energy use (people follow routines). These features will be
used in EDA *and* directly fed into the models in Notebook 2.

Cyclical features (`sin`/`cos` encoding) prevent the model from treating
23:00 and 00:00 as far apart — they are adjacent on the clock.


In [ ]:
df_clean["Year"]       = df_clean.index.year
df_clean["Month"]      = df_clean.index.month
df_clean["Month_Name"] = df_clean.index.month_name()
df_clean["DayOfWeek"]  = df_clean.index.dayofweek
df_clean["Day_Name"]   = df_clean.index.day_name()
df_clean["Hour"]       = df_clean.index.hour
df_clean["IsWeekend"]  = (df_clean.index.dayofweek >= 5).astype(int)
df_clean["Quarter"]    = df_clean.index.quarter
df_clean["Season"]     = df_clean["Month"].map({
    12:"Winter", 1:"Winter", 2:"Winter",
    3:"Spring",  4:"Spring", 5:"Spring",
    6:"Summer",  7:"Summer", 8:"Summer",
    9:"Autumn", 10:"Autumn",11:"Autumn"
})

print("✅ Calendar features added")
df_clean[["Year","Month","DayOfWeek","Hour","IsWeekend","Season"]].head()


## 5. Target Variable Analysis — `Global_active_power`

**Why:** Understanding the distribution of what we're trying to predict
is critical — a right-skewed target often benefits from log-transformation
or tree-based models that handle non-normality natively.

**What we find:**
- **Mean = 1.09 kW**, **Median = 0.60 kW** — large gap signals strong
  right skew (skewness ≈ 1.80)
- Most minutes the household draws < 1 kW (standby loads, lighting)
- Occasional spikes up to **11.1 kW** represent water-heater + oven + HVAC
  running simultaneously
- **Decision:** Use tree-based and LSTM models which handle skew without
  transformation; evaluate with MAE (robust to outliers) alongside RMSE


In [ ]:
gap = df_clean["Global_active_power"]

print("=== Global Active Power — Summary Statistics ===")
stats_dict = {
    "Mean"   : gap.mean(),
    "Median" : gap.median(),
    "Std Dev": gap.std(),
    "Min"    : gap.min(),
    "Max"    : gap.max(),
    "Skew"   : gap.skew(),
    "Kurtosis": gap.kurt(),
    "25th %ile": gap.quantile(0.25),
    "75th %ile": gap.quantile(0.75),
}
for k,v in stats_dict.items():
    print(f"  {k:<12}: {v:.4f} kW")


In [ ]:
fig = px.histogram(
    df_clean.sample(100_000, random_state=42),
    x="Global_active_power",
    nbins=120,
    title="Distribution of Global Active Power — right-skewed, median 0.60 kW",
    labels={"Global_active_power": "Active Power (kW)"},
    color_discrete_sequence=["#1565C0"],
    marginal="box",
)
fig.add_vline(x=gap.mean(),   line_dash="dash", line_color="red",
              annotation_text=f"Mean={gap.mean():.2f}")
fig.add_vline(x=gap.median(), line_dash="dot",  line_color="green",
              annotation_text=f"Median={gap.median():.2f}")
fig.update_layout(title_x=0.5, height=520)
fig.show()


## 6. Time-Series Overview

### 6.1 Full Series (Daily Average)

**Why:** A birds-eye view reveals multi-year trends and confirms seasonality
before we measure it statistically.

**What we learn:**
- Consumption rises every winter and falls every summer — a clean annual cycle
- 2009 shows a slight downward shift mid-year (possible occupancy change or
  energy-saving behaviour)
- No long-term upward trend — this household's baseline usage is stable over
  four years
- **Decision:** Include a weekly seasonal lag (168 h) and a fortnightly lag
  (336 h) in the feature set to capture these repeating patterns


In [ ]:
daily_avg = df_clean["Global_active_power"].resample("D").mean().reset_index()
daily_avg.columns = ["Date", "Avg_Power_kW"]

fig = px.line(
    daily_avg,
    x="Date", y="Avg_Power_kW",
    title="Daily Average Global Active Power — seasonal cycles are clearly visible",
    labels={"Avg_Power_kW": "Avg Power (kW)", "Date": ""},
    color_discrete_sequence=["#1565C0"],
)
fig.update_layout(title_x=0.5, height=450)
fig.show()


### 6.2 Year-over-Year Monthly Comparison

**Why:** Overlaying each year lets us separate *seasonal* effects (same
every year) from *trend* effects (year-on-year change).

**What we learn:**
- The U-shaped winter-summer-winter pattern repeats reliably across all four
  years — strong seasonality we can model
- 2007 peak consumption (December) is the highest on record, while 2010 shows
  a lower winter peak — a mild downward trend exists but is subtle
- **Decision:** Month and quarter features are worth including; a 168 h lag
  covers the weekly cycle but a 336 h lag adds the bi-weekly structure


In [ ]:
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]

monthly = (
    df_clean.groupby(["Year","Month_Name"])["Global_active_power"]
    .mean().reset_index()
)
monthly["Month_Name"] = pd.Categorical(
    monthly["Month_Name"], categories=month_order, ordered=True)
monthly = monthly.sort_values(["Year","Month_Name"])

fig = px.line(
    monthly, x="Month_Name", y="Global_active_power",
    color="Year",
    title="Monthly Average Active Power by Year — stable annual U-shape",
    labels={"Global_active_power": "Avg Power (kW)", "Month_Name": "Month"},
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(title_x=0.5, height=480)
fig.show()


### 6.3 Intraday Pattern — Hour of Day

**Why:** Daily peaks determine *when* appliances are most stressed and
directly help a homeowner decide *when* to run high-energy appliances.

**What we learn:**
- **Night trough (2–5 AM):** 0.3–0.4 kW — essentially standby only
- **Morning ramp (6–9 AM):** Breakfast, showers → rises to ~1.5 kW
- **Evening peak (6–9 PM):** Dinner + TV + evening routines → highest load
- **Weekends** follow the same shape but shifted ~1–2 h later (sleeping in)
  and with a higher midday bump (people at home)
- **Decision for homeowner:** Running dishwasher or washing machine before
  6 AM or after 10 PM avoids peak-load overlap


In [ ]:
hourly = (
    df_clean.groupby(["Hour", "IsWeekend"])["Global_active_power"]
    .mean().reset_index()
)
hourly["Day_Type"] = hourly["IsWeekend"].map({0:"Weekday", 1:"Weekend"})

fig = px.line(
    hourly, x="Hour", y="Global_active_power",
    color="Day_Type",
    title="Average Power by Hour of Day — dual peak at 8 AM and 8 PM",
    labels={"Global_active_power": "Avg Power (kW)", "Hour": "Hour of Day"},
    markers=True,
    color_discrete_sequence=["#1976D2", "#E53935"],
)
fig.update_layout(title_x=0.5, height=450, xaxis=dict(tickmode="linear", dtick=1))
fig.show()


### 6.4 Weekly Pattern — Day of Week

**Why:** Identifies if certain weekdays are consistently higher/lower —
useful for utilities planning demand response and for homeowners scheduling
energy-intensive tasks.

**What we learn:**
- Saturday and Sunday show marginally higher consumption (people at home all day)
- The difference between weekday and weekend is small (~5–8%) compared to
  the seasonal effect — season is the dominant driver
- Error bars (±1 std) are wide, confirming high variability within any given day


In [ ]:
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

weekly = (
    df_clean.groupby("Day_Name")["Global_active_power"]
    .agg(["mean","std"]).reindex(day_order).reset_index()
)

fig = go.Figure([go.Bar(
    x=weekly["Day_Name"], y=weekly["mean"],
    error_y=dict(type="data", array=weekly["std"], visible=True, color="#999"),
    marker_color=["#42A5F5"]*5 + ["#EF5350","#EF5350"],
)])
fig.update_layout(
    title="Average Active Power by Day — weekends slightly higher, variance is large",
    xaxis_title="Day of Week", yaxis_title="Avg Power (kW)",
    title_x=0.5, height=450, showlegend=False
)
fig.show()


### 6.5 Seasonal Boxplot

**Why:** Boxplots show both the *central tendency* and *spread* per season,
revealing whether the higher winter average is driven by everyone using more
or by occasional extreme events.

**What we learn:**
- Winter median ≈ 0.85 kW vs Summer median ≈ 0.45 kW — a ~90% lift
- Winter IQR is also wider — more variable consumption (some days mild heating,
  others full heating + cooking)
- Summer has the tightest spread — predictable air-conditioning pattern
- **Decision:** Season flag and month features are valuable; winter errors in
  the model will likely be higher (confirmed in Notebook 2 error analysis)


In [ ]:
df_sample = df_clean[df_clean["Season"].isin(["Winter","Spring","Summer","Autumn"])]    .sample(50_000, random_state=42)

fig = px.box(
    df_sample, x="Season", y="Global_active_power",
    category_orders={"Season":["Winter","Spring","Summer","Autumn"]},
    color="Season",
    title="Power Distribution by Season — winter median 2× summer median",
    labels={"Global_active_power": "Active Power (kW)"},
    color_discrete_map={
        "Winter":"#1E88E5","Spring":"#43A047",
        "Summer":"#FB8C00","Autumn":"#8E24AA"
    },
)
fig.update_layout(title_x=0.5, height=480, showlegend=False)
fig.show()


## 7. Sub-Meter Contribution Analysis

**Why:** The dataset tracks three sub-meters. Understanding their relative
contribution tells us whether they help predict the target *or* are largely
redundant noise.

| Sub-meter | Appliances |
|-----------|-----------|
| Sub_metering_1 | Kitchen (microwave, dishwasher, oven) |
| Sub_metering_2 | Laundry (washing machine, tumble dryer, fridge) |
| Sub_metering_3 | Water heater + air conditioner |

**What we find:**
- **HVAC (Sub3): 0.59%** of total energy — the largest metered sub-load
- **Laundry (Sub2): 0.12%**
- **Kitchen (Sub1): 0.10%**
- **Unmetered / other: 99.2%** — lighting, TV, computers, EV charger, etc.

**Key insight:** The three sub-meters together explain only 0.8% of total
consumption. This means the *dominant driver of total power* is unmetered
loads — the sub-meter readings are weak direct predictors of the global target.
However, their temporal patterns (e.g., HVAC cycling) may still help capture
usage rhythms, so we retain them in the feature set.


In [ ]:
# Convert sub-metering from Wh/min to kW equivalent for direct comparison
df_clean["Sub1_kW"] = df_clean["Sub_metering_1"] / 60
df_clean["Sub2_kW"] = df_clean["Sub_metering_2"] / 60
df_clean["Sub3_kW"] = df_clean["Sub_metering_3"] / 60

total_energy = df_clean["Global_active_power"].sum()
shares = {
    "Kitchen (Sub1)" : df_clean["Sub1_kW"].sum() / total_energy * 100,
    "Laundry (Sub2)" : df_clean["Sub2_kW"].sum() / total_energy * 100,
    "HVAC    (Sub3)" : df_clean["Sub3_kW"].sum() / total_energy * 100,
}
shares["Unmetered/Other"] = 100 - sum(shares.values())

print("=== Energy Share per Sub-meter ===")
for k,v in shares.items():
    print(f"  {k:<20}: {v:.3f}%")


In [ ]:
fig = px.pie(
    names=list(shares.keys()),
    values=list(shares.values()),
    title="Energy Share by Sub-meter — 99.2% is unmetered (lighting, TV, computers…)",
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4
)
fig.update_layout(title_x=0.5, height=480)
fig.show()


In [ ]:
# Monthly sub-meter trend — is HVAC seasonal?
monthly_sub = (
    df_clean.resample("ME")[["Sub1_kW","Sub2_kW","Sub3_kW"]]
    .mean().reset_index()
    .melt(id_vars="Datetime", var_name="Sub_Meter", value_name="Avg_kWh_per_min")
)
monthly_sub["Sub_Meter"] = monthly_sub["Sub_Meter"].map({
    "Sub1_kW":"Kitchen","Sub2_kW":"Laundry","Sub3_kW":"HVAC"
})

fig = px.line(
    monthly_sub, x="Datetime", y="Avg_kWh_per_min",
    color="Sub_Meter",
    title="Monthly Sub-meter Trends — HVAC spikes in winter (heating), dips in summer",
    labels={"Avg_kWh_per_min": "Avg Consumption (kWh/min)", "Datetime": ""},
    color_discrete_sequence=["#E53935","#43A047","#1E88E5"],
)
fig.update_layout(title_x=0.5, height=450)
fig.show()


## 8. Correlation Analysis

**Why:** A Pearson correlation matrix quantifies *linear* relationships between
all sensor variables. This guides feature selection and flags multicollinearity.

**What we find:**
- `Global_active_power` ↔ `Global_intensity`: **r = 1.00** — perfect linear
  relationship (P = V × I, and voltage is nearly constant at ~241 V). These
  two columns are *identical predictors*; we use only power as the target
- `Global_active_power` ↔ `Sub_metering_3` (HVAC): **r = 0.64** — moderate
  positive; HVAC is the only sub-meter worth including as a raw feature
- `Voltage` ↔ `Global_intensity`: **r = −0.40** — mild negative; as demand
  rises, the mains voltage sags slightly (Ohm's law)
- Sub_metering_1 and Sub_metering_2 show weak correlations with the target
  (r ≈ 0.10–0.12), confirming they contribute little direct explanatory power


In [ ]:
corr_cols = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"
]
corr = df_clean[corr_cols].corr()

fig = px.imshow(
    corr, text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Pearson Correlation Matrix — intensity and active power are co-linear",
)
fig.update_layout(title_x=0.5, height=600)
fig.show()


## 9. Autocorrelation & Lag Structure

**Why:** Autocorrelation (ACF) tells us *how far back* past power values
remain predictive of the current value. This directly justifies which lag
features to create in Notebook 2.

**What we find (from the hourly-resampled series):**
- Lag 1 h: ACF ≈ 0.71 — very strong; the last hour is the best single predictor
- Lag 24 h: ACF ≈ 0.55 — same hour yesterday is also predictive (daily cycle)
- Lag 168 h: ACF ≈ 0.48 — same hour last week (weekly cycle)
- ACF remains positive and significant out to ~336 h (two weeks)

**Decision:** Create lag features at 1, 2, 3, 6, 12, 24, 48, 72, 168, and
336 hours. The significant weekly autocorrelation is why a naïve lag-168h
baseline performs better than a random predictor.


In [ ]:
# Compute ACF manually (no statsmodels needed)
hourly_gap = df_clean["Global_active_power"].resample("h").mean().dropna()
max_lag = 24 * 14 + 1   # 14 days

acf_values = [hourly_gap.autocorr(lag=l) for l in range(1, max_lag)]
acf_df = pd.DataFrame({"Lag_h": range(1, max_lag), "ACF": acf_values})

# Highlight key lag points
key_lags  = [1, 24, 48, 72, 168, 336]
key_flags = acf_df["Lag_h"].isin(key_lags)

fig = go.Figure()
fig.add_trace(go.Scatter(x=acf_df["Lag_h"], y=acf_df["ACF"],
                          mode="lines", line=dict(color="#1565C0"),
                          name="ACF"))
fig.add_trace(go.Scatter(
    x=acf_df.loc[key_flags,"Lag_h"],
    y=acf_df.loc[key_flags,"ACF"],
    mode="markers+text",
    marker=dict(color="#E53935", size=9),
    text=[f"{l}h" for l in key_lags],
    textposition="top center",
    name="Key lags",
))
conf_bound = 1.96 / np.sqrt(len(hourly_gap))
fig.add_hline(y= conf_bound, line_dash="dot", line_color="grey")
fig.add_hline(y=-conf_bound, line_dash="dot", line_color="grey")
fig.update_layout(
    title="Autocorrelation Function (ACF) — hourly active power (lags up to 14 days)",
    xaxis_title="Lag (hours)", yaxis_title="Autocorrelation",
    title_x=0.5, height=450
)
fig.show()

print(f"\nKey ACF values:")
for l in key_lags:
    v = hourly_gap.autocorr(lag=l)
    print(f"  Lag {l:>3}h : {v:.4f}")


## 10. Outlier Detection

**Why:** Extreme values can distort training if a model memorises them.
We detect outliers with two methods and decide whether to remove them.

**What we find:**
- **Z-score (|z| > 3):** 36,803 readings (1.77%) — high-power events like
  simultaneous oven + water heater + air conditioner
- **IQR fences:** 95,699 readings (4.61%) — more generous bounds

**Decision:** We do **not** remove outliers. These spikes represent real
household behaviour (cooking large meals, parties). Removing them would make
the model underestimate peak consumption — the exact scenario that matters most
for demand management. Instead, we use tree-based models (Random Forest,
XGBoost, LightGBM) that are robust to extreme values.


In [ ]:
gap_clean = df_clean["Global_active_power"].dropna()

z       = np.abs(stats.zscore(gap_clean))
z_out   = (z > 3).sum()
Q1, Q3  = gap_clean.quantile([0.25, 0.75])
IQR     = Q3 - Q1
iqr_out = ((gap_clean < Q1 - 1.5*IQR) | (gap_clean > Q3 + 1.5*IQR)).sum()

print(f"Z-score outliers (|z|>3): {z_out:,}  ({z_out/len(gap_clean)*100:.2f}%)")
print(f"IQR     outliers        : {iqr_out:,}  ({iqr_out/len(gap_clean)*100:.2f}%)")
print(f"\n→ Decision: RETAIN all outliers — they represent real peak usage events.")


In [ ]:
corr_cols = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"
]
fig = px.box(
    df_clean[corr_cols].sample(30_000, random_state=42)
        .melt(var_name="Feature", value_name="Value"),
    x="Feature", y="Value",
    color="Feature",
    title="Boxplot of All Numeric Features (30 k sample) — Voltage is most stable",
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_layout(title_x=0.5, height=500, showlegend=False)
fig.show()


## 11. Save Processed Datasets for Notebook 2

We create three derived datasets:
- `hourly_clean.csv` — 1-hour averages (main modelling dataset)
- `daily_clean.csv`  — 1-day averages (trend visualisation)
- `minute_clean.parquet` — full minute-level data (used by Streamlit app)


In [ ]:
numeric_cols = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"
]
df_hourly = df_clean[numeric_cols].resample("h").mean()
df_daily  = df_clean[numeric_cols].resample("D").mean()

print(f"Hourly dataset : {df_hourly.shape}")
print(f"Daily  dataset : {df_daily.shape}")

df_hourly.to_csv("../data/hourly_clean.csv")
df_daily.to_csv("../data/daily_clean.csv")
df_clean.to_parquet("../data/minute_clean.parquet")

print("\n✅ Saved hourly_clean.csv | daily_clean.csv | minute_clean.parquet")


## 12. Key EDA Findings & Decisions for Modelling

| # | Finding | Impact on Modelling |
|---|---------|-------------------|
| 1 | **2,075,259** minute-level records; 1.25% missing (block gaps) | Use `ffill`; not problematic for hourly aggregation |
| 2 | Target is **right-skewed** (skew ≈ 1.80, mean > median) | Use tree models + MAE metric; don't log-transform |
| 3 | Strong **annual seasonality** (winter ~2× summer) | Include month, quarter, season features |
| 4 | Clear **daily cycle**: dual peaks at 8 AM and 8 PM | Include hour + cyclical sin/cos encoding |
| 5 | **Weekly cycle**: ACF significant at 168 h lag | Include lag_168h and lag_336h features |
| 6 | lag_1h ACF = 0.71 — strongest single predictor | lag_1h will dominate feature importance |
| 7 | Sub-meters explain only **0.8%** of total energy | Sub-meters are weak direct predictors |
| 8 | `Global_intensity` perfectly colinear with target | Drop from model input (or keep — will score zero importance) |
| 9 | **HVAC (Sub3)** is the biggest metered load (0.59%) | Sub3 worth retaining in feature set |
| 10 | Outliers (1.77%) represent real peak events | Retain; use robust models + MAE metric |

